# Tutorial 20: SSA and Conjunction Assessment

This tutorial demonstrates how to perform Space Situational Awareness (SSA) tasks, including TLE-to-TLE conjunction assessment and planning collision avoidance (COLA) maneuvers.

## 1. Conjunction Assessment
A conjunction occurs when two space objects pass within a defined safety volume. We use the `conjunction` module to find the Time of Closest Approach (TCA).

In [3]:
import numpy as np
from datetime import datetime, timedelta
from opengnc.ssa.conjunction import find_tca, compute_pc_chan
from opengnc.ssa.tle_interface import TLEEntity
from opengnc.propagators.sgp4_propagator import Sgp4Propagator

# 1. Define two satellites using TLEs (simplified ISS data and debris)
tle1 = TLEEntity("ISS", 
                 "1 25544U 98067A   24001.00000000  .00016717  00000-0  10270-3 0  9011",
                 "2 25544  51.6416 247.4627 0006703 130.5360 325.0288 15.50030391432358")
tle2 = TLEEntity("DEBRIS", 
                 "1 99999U 24001A   24001.00000000  .00000000  00000-0  00000-0 0  9990",
                 "2 99999  51.6420 247.4630 0006700 130.5000 325.0000 15.50000000000000")

# 2. Setup Propagators
prop1 = tle1.get_propagator()
prop2 = tle2.get_propagator()

# Define position functions for find_tca (takes time in seconds from TLE epoch)
# Note: Sgp4Propagator.propagate requires dummy r_i, v_i as they are defined in the base class
def pos1(t): return prop1.propagate(np.zeros(3), np.zeros(3), t)[0]
def pos2(t): return prop2.propagate(np.zeros(3), np.zeros(3), t)[0]

# 3. Find TCA within a 24-hour window
tca = find_tca(pos1, pos2, 0, 86400)
dist = np.linalg.norm(pos1(tca) - pos2(tca))

print(f"Time of Closest Approach (TCA): {tca:.2f} seconds after epoch")
print(f"Minimum Separation Distance: {dist / 1000.0:.3f} km")

Time of Closest Approach (TCA): 0.04 seconds after epoch
Minimum Separation Distance: 7.665 km


## 2. Collision Avoidance Maneuver (COLA)
If the distance is below a threshold (e.g., 5 km), a maneuver is required. We can plan an impulsive Delta-V pulse to minimize the probability of collision (Pc).

In [ ]:
from opengnc.ssa.collision_avoidance import plan_collision_avoidance_maneuver

# 1. Get states at TCA for both objects
r1, v1 = pos1(tca), prop1.propagate(np.zeros(3), np.zeros(3), tca)[1]
r2, v2 = pos2(tca), prop2.propagate(np.zeros(3), np.zeros(3), tca)[1]

# 2. Assume some covariances (Diagonal 100m - 200m sigma)
cov1 = 100.0**2 * np.eye(3)
cov2 = 200.0**2 * np.eye(3)

# 3. Plan maneuver 12 hours (43200s) before TCA
dv, final_pc = plan_collision_avoidance_maneuver(
    r1, v1, cov1, 
    r2, v2, cov2, 
    hbr=10.0,            # Combined Hard Body Radius (m)
    t_man_before_tca=43200, 
    pc_limit=1e-6
)

print(f"Required COLA Delta-V Vector (m/s): {dv}")
print(f"Total Magnitude: {np.linalg.norm(dv):.4f} m/s")
print(f"Final Pc: {final_pc:.2e}")

Required COLA Delta-V Vector (m/s): [-1.28871853e-05 -3.66698348e-05  4.52790680e-06]
Total Magnitude: 0.0000 m/s
Final Pc: 3.94e-259
